# Скоринг горнолыжных курортов

## Оглавление
1. [Расчет баллов для ранжирования горнолыжных курортов](#Расчет-баллов-для-ранжирования-горнолыжных-курортов)
2. [Итоговый рейтинг](#Итоговый-рейтинг)
3. [Сценарный рейтинг](#Сценарный-рейтинг)
4. [Ответы на вопросы для исследования согласно ТЗ](#Ответы-на-вопросы-для-исследования-согласно-ТЗ)

## Расчет баллов для ранжирования горнолыжных курортов

### Подготовка окружения

In [1]:
import pandas as pd
import numpy as np

mart = pd.read_csv('../data/mart/resort_features.csv')
print(mart.shape)
mart.head(3)

(32, 43)


,resort_id,name_ru,region,latitude,longitude,coordinate_quality,operating,status_note,piste_total_km_6km,lift_lines_6km,...,freeride,tubing,adult_price_min_rub,adult_price_max_rub,price_season,source_confidence,trips_2025,trips_growth_2025,nights_2025,avg_stay_nights_2025
0,R001,Роза Хутор,Краснодарский край,43.646390,40.332220,checked_ok,yes,NaN,159.4,64,...,1.0,1.0,NaN,NaN,2026/27,official_single_source,18427326.0,-0.038392,110670767,6.005796
1,R002,Красная Поляна,Краснодарский край,43.677244,40.262795,official_verified,yes,NaN,93.2,58,...,1.0,1.0,NaN,NaN,2025/26,not_specified,18427326.0,-0.038392,110670767,6.005796
2,R003,Газпром Поляна,Краснодарский край,43.699608,40.313290,official_verified,yes,NaN,73.5,46,...,1.0,1.0,3600.0,3600.0,2025/26,not_specified,18427326.0,-0.038392,110670767,6.005796


### Словарь скоринга
Ранжирование будет основываться на двух категориях показателей.

#### Базовые показатели и веса согласно ТЗ для составления базового рейтинга

| Показатель | Колонка витрины | Шкала | Почему такая шкала | Вес ТЗ |
|---|---|---|---|---|
| Поездки по региону 2025 | trips_2025 | log min max | Логарифм уменьшает влияние единичных очень больших значений | 25 |
| Рост поездок | trips_growth_2025 | Линейная шкала с ограничением | Формула не зависит от выборки | 10 |
| Средняя длительность | avg_stay_nights_2025 | Линейная шкала с ограничением | Формула не зависит от выборки | 5 |
| Трассы OSM | piste_total_km_6km | log min max | Логарифм уменьшает влияние единичных очень больших значений | 10 |
| Подъемники OSM | lift_lines_6km | log min max | Логарифм уменьшает влияние единичных очень больших значений | 7 |
| Размещение OSM | lodging_objects_6km | log min max | Логарифм уменьшает влияние единичных очень больших значений | 5 |
| Питание и места интереса OSM | food_objects_6km + attraction_objects_6km | log min max | Логарифм уменьшает влияние единичных очень больших значений | 3 |
| Снежные дни | avg_snow_days_per_season | min max | Обычная нормализация | 8 |
| Морозные дни | avg_freezing_days_per_season | min max | Обычная нормализация | 6 |
| Сезонный снегопад | avg_snowfall_cm_per_season | log min max | Логарифм уменьшает влияние единичных очень больших значений | 6 |
| Расстояние до аэропорта | nearest_airport_distance_km | Обратная log min max  | Чем меньше значение, тем лучше | 8 |
| Локальный транспорт OSM | transport_objects_6km | log min max | Логарифм уменьшает влияние единичных очень больших значений | 2 |
| Цена дневного ски-пасса (минимальный дневной тариф без льгот) | adult_price_min_rub | Обратная log min max  | Чем меньше значение, тем лучше | 5 |


#### Сценарные показатели для составления опционального рейтинга

| Показатель | Колонка витрины | Шкала | Почему такая шкала |
|---|---|---|---|
| Солнечные часы | winter_mean_sunshine_hours | порог 6 | 6 часов солнца в зимний день = комфортно | 
| Расстояние до жд-станции | nearest_railway_station_distance_km | Обратная log min max  | ближе станция - проще добраться | 
| Зеленые трассы | green | порог 10 | 10 зеленых = достаточный выбор для новичков | 
| Синие трассы | blue | порог 15 | синих у всех много, "достаточно" больше |
| Красные трассы | red | порог 10 | средний вариант |
| Черные трассы | black | порог 5 | черные - редкость, 5 = серьезная гора |
| Наличие фрирайда | freeride | 0/1 × 100 | бинарный признак: есть или нет |
| Наличие тюбинг трассы | tubing | 0/1 × 100 | бинарный признак: есть или нет |
| Перепад высот курорта (общий перепад зоны катания) | drop_m | порог 850 | 850 м = 85-й перцентиль, «достаточно эпично»; наш порог по правилу насыщения |


### Подготовка функций для расчета баллов
* Скоринг: сырые показатели → баллы 0–100.
* Вход: витрина data/mart/resort_features.csv (32 курорта).

#### Шкала min max
* Минимум выборки получает 0, максимум - 100.
* Используется для снежных и морозных дней.
* Тестируем на условных числах, прежде чем пускать на курорты.

In [2]:
# Суть: кладем колонку сырых чисел - получаем колонку баллов 0-100
def min_max_score(column):
    x_min = column.min()
    x_max = column.max()
    return 100 * (column - x_min) / (x_max - x_min)


# Тест на условных числах: одно крошечное, одно среднее, одно гигантское.
test = pd.Series([0, 10, 100, 1000])
min_max_score(test)

0      0.0
1      1.0
2     10.0
3    100.0
dtype: float64

#### Шкала log min max
* Логарифм уменьшает влияние единичных очень больших значений.
* Ноль допустим, потому что перед логарифмом к показателю прибавляется единица.
* Сначала пропускаем числа через ln(1+x) - он сжимает гигантов.
* Затем та же функция, что и для min max.
* Используется для поездок, OSM объектов, локального транспорта и сезонного снегопада.

In [3]:
# ln(1+x): единица внутри, чтобы ноль не ломал логарифм (ln(0) не существует)
def log_min_max_score(column):
    log_values = np.log(1 + column)
    return min_max_score(log_values)

# Тот же тест на тех же числах
log_min_max_score(test)

0      0.000000
1     34.708068
2     66.801047
3    100.000000
dtype: float64

#### Шкала обратная log min max
* Для признаков, где меньше = лучше.
* Сначала обычный логарифмический балл, потом разворот.
* Первый становится последним.
* Используется для расстояния до аэропорта и жд-вокзала и цены.

In [4]:
# Обратная шкала: берем готовый лог-балл и разворачиваем его
def inverse_score(column):
    return (100 - log_min_max_score(column)).clip(0, 100)

# Тот же тест: ряд как будто "расстояния в км" - кто ближе, тот лучше
inverse_score(test)

0    100.000000
1     65.291932
2     33.198953
3      0.000000
dtype: float64

#### Линейная шкала с ограничением
* Для роста и ночевок ТЗ дает готовые формулы с обрезкой краев: рост −25% и хуже = 0, +25% и выше = 100, нулевой рост = 50; ночи: 1 ночь = 0, 7 ночей и больше = 100.
* Эти формулы не зависят от выборки.
* Используется для роста поездок и ночевок.

In [5]:
# clip(0, 100) обрезает все, что вылезло за края, - страховка от аномалий.
def growth_score(growth):
    return (50 + 200 * growth).clip(0, 100)

def stay_score(nights):
    return ((nights - 1) / 6 * 100).clip(0, 100)

# Тест роста: впадина, граница, Краснодарский край, ноль, потолок, аномалия
test_growth = pd.Series([-0.5, -0.25, -0.0384, 0, 0.25, 0.86])
print(growth_score(test_growth))

# Тест ночевок: минимум, медиана наших данных, Краснодарский край, потолок
test_stay = pd.Series([1, 3.56, 6.006, 7.83])
print(stay_score(test_stay))

0      0.00
1      0.00
2     42.32
3     50.00
4    100.00
5    100.00
dtype: float64
0      0.000000
1     42.666667
2     83.433333
3    100.000000
dtype: float64


#### Пороговая шкала
* Переводит значение в баллы не сравнением с другими курортами, а сравнением с уровнем "достаточно хорошо": дошел до порога - 100, не дошел - доля пути.
* Перевыполнение не добавляет баллов.
* Используется для сценарных показателей: сложность трасс, солнечные часы, перепад высот.

In [6]:
def threshold_score(column, threshold):
    return (100 * column / threshold).clip(0, 100)

# Тест на контрольных точках: зеленые трассы, порог = 10
# 0 - трасс нет, 2 - медиана наших зеленых, 5 - половина порога
# 10 - сам порог, 16 - Манжерок
threshold_score(pd.Series([0, 2, 5, 10, 16]), 10)
# ожидание: 0 / 20 / 50 / 100 / 100

0      0.0
1     20.0
2     50.0
3    100.0
4    100.0
dtype: float64

### Применение функций для расчета баллов
* Суффикс _score означает, что это уже балл, не сырые данные.
* В заголовках блоков - функция, в скобках - смысловая группа показателей из ТЗ.

#### Шкала min-max
* Ровные величины без гигантов.
* Снежные и морозные дни: разброс ровный (25–107 и 27–180) - обычная нормализация.

In [7]:
# scores - новая таблица: только курорт и баллы, сырые числа остаются в mart
scores = pd.DataFrame({
    'resort_id': mart['resort_id'],
    'name_ru': mart['name_ru'],
    'snow_days_score': min_max_score(mart['avg_snow_days_per_season']),
    'frost_days_score': min_max_score(mart['avg_freezing_days_per_season']),
})

print(scores.shape)
print("Топ-5 по снежным дням:")
print(scores.sort_values("snow_days_score", ascending=False)
             .head(5)[["name_ru", "snow_days_score"]]
             .to_string(index=False))

print("\nТоп-5 по морозным дням:")
print(scores.sort_values("frost_days_score", ascending=False)
             .head(5)[["name_ru", "frost_days_score"]]
             .to_string(index=False))

(32, 4)
Топ-5 по снежным дням:
        name_ru  snow_days_score
Большой Вудъявр       100.000000
   Кукисвумчорр        97.066015
         Домбай        95.110024
  Морозная гора        92.420538
        Эльбрус        90.709046

Топ-5 по морозным дням:
        name_ru  frost_days_score
Большой Вудъявр        100.000000
        Эльбрус         95.448635
   Кукисвумчорр         91.157347
  Морозная гора         88.686606
          Чегет         87.386216


##### Проверка: шкала min-max
* Все баллы блока в 0-100.
* Пропусков нет - климат посчитан для всех 32.
* Большой Вудъявр и Кукисвумчорр явные лидеры по двум показателям сразу - обе горнолыжки находятся в Мурманской области.

#### Шкала log min max
* "Питание и места интереса" - одной суммой, потому что в ТЗ у группы один вес.

In [8]:
# Региональный спрос
scores['trips_score'] = log_min_max_score(mart['trips_2025'])

# Инфраструктура
scores['trails_km_score'] = log_min_max_score(mart['piste_total_km_6km'])
scores['lifts_score'] = log_min_max_score(mart['lift_lines_6km'])
scores['lodging_score'] = log_min_max_score(mart['lodging_objects_6km'])
scores['food_score'] = log_min_max_score(mart['food_objects_6km'] + mart['attraction_objects_6km'])

# Сезонный снегопад и локальный транспорт
scores['snowfall_score'] = log_min_max_score(mart['avg_snowfall_cm_per_season'])
scores['transport_score'] = log_min_max_score(mart['transport_objects_6km'])

print(scores.shape)
print("Топ-5 по региональному спросу:")
print(scores.sort_values("trips_score", ascending=False)
             .head(5)[["name_ru", "trips_score"]]
             .to_string(index=False))

scores["infrastructure_score"] = (
    scores["trails_km_score"]
    + scores["lifts_score"]
    + scores["lodging_score"]
    + scores["food_score"]
)

print("\nТоп-5 по инфраструктуре:")
print(scores.sort_values("infrastructure_score", ascending=False)
             .head(5)[["name_ru", "infrastructure_score"]]
             .to_string(index=False))

print("Топ-5 по сезонному снегопаду:")
print(scores.sort_values("snowfall_score", ascending=False)
             .head(5)[["name_ru", "snowfall_score"]]
             .to_string(index=False))

print("Топ-5 по локальному транспорту:")
print(scores.sort_values("transport_score", ascending=False)
             .head(5)[["name_ru", "transport_score"]]
             .to_string(index=False))

(32, 11)
Топ-5 по региональному спросу:
       name_ru  trips_score
      Сорочаны   100.000000
    Роза Хутор    95.844308
Газпром Поляна    95.844308
Красная Поляна    95.844308
         Игора    80.426525

Топ-5 по инфраструктуре:
       name_ru  infrastructure_score
Красная Поляна            383.730513
    Роза Хутор            363.400916
Газпром Поляна            358.834975
       Шерегеш            324.566160
         Архыз            306.641389
Топ-5 по сезонному снегопаду:
      name_ru  snowfall_score
       Домбай      100.000000
      Эльбрус       91.283421
        Чегет       88.706640
Морозная гора       84.176419
          Цей       69.247161
Топ-5 по локальному транспорту:
        name_ru  transport_score
   Бобровый лог       100.000000
  Горный воздух        91.840274
       Сорочаны        77.839053
Большой Вудъявр        77.258296
  КолаСпортЛэнд        76.960021


##### Проверка: шкала log min max
* Все баллы блока в 0-100.
* Пропусков нет - OSM и спрос посчитаны для всех 32.
* Сорочаны являются явным лидером по региональному спросу благодаря Москве и МО, краснодарская тройка следующие топ 2-4.
* По инфраструктуре топ-3 - Краснодарская тройка.
* По сезонному снегопаду лидирует Домбай, Кировские горнолыжки не попали в топ-5.
* По локальному транспорту также лидируют курорты, находящиеся в близости к городу.

#### Шкала обратная log min max
* Аэропорт и цена - базовые показатели; расстояние до ЖД-станции - сценарный (в базовый рейтинг не входит, живет в отдельной таблице scenario_scores - создаем ее здесь: это первый сценарный балл по ходу функций).
* За цену ски-пасса взят минимальный полный дневной взрослый тариф без льгот.
* Особые случаи цены - пояснение ниже.

In [9]:
# Доступность: ближе аэропорт - больше балл
scores['airport_score'] = inverse_score(mart['nearest_airport_distance_km'])

# Цена: дешевле = лучше
scores['price_score'] = inverse_score(mart['adult_price_min_rub'])

# Сценарный: расстояние до ЖД-станции (ближе - лучше)
scenario_scores = pd.DataFrame({
    'resort_id': mart['resort_id'],
    'name_ru': mart['name_ru'],
    'railway_score': inverse_score(mart['nearest_railway_station_distance_km']),
})

# Восьми горным курортам станция не нашлась в радиусе сбора 50 км:
# это "дальше 50 км", а не пропуск - честный балл 0
scenario_scores['railway_score'] = scenario_scores['railway_score'].fillna(0)

print(scores.shape, scenario_scores.shape)

print("Топ-3 по доступности аэропорта:")
print(scores.sort_values("airport_score", ascending=False)
             .head(3)[["name_ru", "airport_score"]]
             .to_string(index=False))

print("Топ-3 по цене:")
print(scores.sort_values("price_score", ascending=False)
             .head(3)[["name_ru", "price_score"]]
             .to_string(index=False))

print("Топ-3 по жд-станции:")
print(scenario_scores.sort_values("railway_score", ascending=False)
             .head(3)[["name_ru", "railway_score"]]
             .to_string(index=False))

(32, 14) (32, 3)
Топ-3 по доступности аэропорта:
       name_ru  airport_score
 Горный воздух     100.000000
 Морозная гора      95.160815
Красная Глинка      91.186787
Топ-3 по цене:
        name_ru  price_score
            Цей   100.000000
Большой Вудъявр   100.000000
        Евразия    62.639869
Топ-3 по жд-станции:
       name_ru  railway_score
Красная Глинка     100.000000
        Губаха      95.008905
Красная Поляна      92.063418


##### Проверка: шкала обратная log min max
* Все баллы обеих таблиц в 0-100.
* Ожидаем пропуски в price_score - курорты без дневной цены (особые случаи ниже).
* railway_score пропусков не имеет: "нет станции в 50 км" уже переведено в честный 0.
* Абсолютный лидер по близости к аэропорту - Горный воздух на Сахалине.
* По цене топ делят Цей и Большой Вудъявр, однако держим в уме, что не все курорты получилось привести к единому тарифу.
* Близость к городу помогает Красная Глинка набрать 100 баллов по жд доступности.

##### Особые случаи: цена
Пропуск цены → балл 50. У 3 курортов нет цен из-за статуса работы (Телецкий, Аджигардак - реконструкция, КолаСпортЛэнд - закрыт), у 10 курортов нет цен из-за отсутствия дневного тарифа: по ТЗ §12 их ценовой балл = 50 (нейтральная середина), а сам факт пропуска останется виден в витрине и потом в quality_flags.

In [10]:
scores['price_score'] = scores['price_score'].fillna(50)

# Кто именно получил нейтраль - проверяем по витрине, там честный пропуск еще виден
mart.loc[mart['adult_price_min_rub'].isna(), ['resort_id', 'name_ru', 'operating']]

,resort_id,name_ru,operating
0,R001,Роза Хутор,yes
1,R002,Красная Поляна,yes
8,R009,Мамисон,yes
11,R012,Телецкий,reconstruction
14,R015,КолаСпортЛэнд,closed
15,R016,Горный воздух,yes
18,R019,Бобровый лог,yes
22,R023,Аджигардак,reconstruction
23,R024,Солнечная долина,yes
24,R025,Завьялиха,yes


#### Линейная шкала с ограничением
* Рост: −25%..+25% (формула 50 + 200×рост, клип 0-100).
* Длительность: от 1 до 7 ночей ((ночи − 1)/6).
* Границы заданы смыслом, а не выборкой.

In [11]:
# Рост поездок и средняя длительность
scores['growth_score'] = growth_score(mart['trips_growth_2025'])
scores['stay_score'] = stay_score(mart['avg_stay_nights_2025'])

print(scores.shape)

print("Топ-5 по росту поездок:")
print(scores.sort_values("growth_score", ascending=False)
             .head(5)[["name_ru", "growth_score"]]
             .to_string(index=False))

print("Топ-3 по средней длительности:")
print(scores.sort_values("stay_score", ascending=False)
             .head(3)[["name_ru", "stay_score"]]
             .to_string(index=False))

(32, 16)
Топ-5 по росту поездок:
      name_ru  growth_score
       Домбай    100.000000
        Архыз    100.000000
   Гладенькая    100.000000
Морозная гора    100.000000
     Сорочаны     99.100406
Топ-3 по средней длительности:
   name_ru  stay_score
   Эльбрус  100.000000
     Чегет  100.000000
Роза Хутор   83.429939


##### Проверка: линейная шкала с ограничением
* Все баллы блока в 0-100.
* Пропусков нет - региональные метрики есть у всех 32.
* Сразу 4 курорта имеют 100 баллов по росту поездок: Домбай, Архыз, Гладенькая и Морозная гора.
* Эльбрус и Чегет делят топ по средней длительности поездок.

#### Пороговая шкала
* Цвета трасс, солнечные часы и перепад - сценарные показатели: живут в scenario_scores, в базовый рейтинг не входят. 

In [12]:
# Солнце: в витрине его нет - пристегиваем merge-ом из климатической таблицы по resort_id
climate = pd.read_csv('../data/prepared/resort_climate_features.csv')
sun = climate[['resort_id', 'winter_mean_sunshine_hours']]
scenario_scores = scenario_scores.merge(sun, on='resort_id', how='left', validate='one_to_one')

# Солнечные часы: 6 часов в зимний день
scenario_scores['sun_score'] = threshold_score(scenario_scores['winter_mean_sunshine_hours'], 6)

# Часы солнца были нужны только для балла - убираем, в таблице остаются только баллы
scenario_scores = scenario_scores.drop(columns=['winter_mean_sunshine_hours'])

print("Список курортов с сортировкой по солнечным дням:")
print(scenario_scores.sort_values("sun_score", ascending=False)
             .head(32)[["name_ru", "sun_score"]]
             .to_string(index=False))

# Цвета трасс
scenario_scores['green_score'] = threshold_score(mart['green'], 10)   # семья: где учиться новичку
scenario_scores['blue_score'] = threshold_score(mart['blue'], 15)
scenario_scores['red_score'] = threshold_score(mart['red'], 10)
scenario_scores['black_score'] = threshold_score(mart['black'], 5)    # экспертные трассы

scenario_scores["tr_score"] = (
    scenario_scores["green_score"]
    + scenario_scores["blue_score"]
    + scenario_scores["red_score"]
    + scenario_scores["black_score"]
)

print("\nТоп-5 по трассам:")
print(scenario_scores.sort_values("tr_score", ascending=False)
             .head(5)[["name_ru", "tr_score"]]
             .to_string(index=False))


# Перепад высот курорта (общий перепад зоны катания): порог 850 м 
scenario_scores['drop_score'] = threshold_score(mart['drop_m'], 850)

print("Список курортов с сортировкой по перепаду высот:")
print(scenario_scores.sort_values("drop_score", ascending=False)
             .head(32)[["name_ru", "drop_score"]]
             .to_string(index=False))

Список курортов с сортировкой по солнечным дням:
         name_ru  sun_score
      Роза Хутор 100.000000
  Красная Поляна 100.000000
  Газпром Поляна 100.000000
           Архыз 100.000000
         Эльбрус 100.000000
          Домбай 100.000000
           Чегет 100.000000
         Мамисон 100.000000
   Горный воздух 100.000000
             Цей 100.000000
        Манжерок 100.000000
        Телецкий 100.000000
Солнечная долина 100.000000
  Соболиная гора 100.000000
   Морозная гора 100.000000
         Холдоми 100.000000
        Мраткино 100.000000
      Гладенькая 100.000000
        Абзаково 100.000000
         Шерегеш  99.758830
    Бобровый лог  97.414459
       Завьялиха  93.575975
         Евразия  87.083701
  Красная Глинка  81.718727
      Аджигардак  81.450883
      Гора Белая  80.417403
           Игора  68.121413
        Сорочаны  67.407101
          Губаха  63.929360
 Большой Вудъявр  54.402134
   КолаСпортЛэнд  54.402134
    Кукисвумчорр  51.750184

Топ-5 по трассам:
        

##### Проверка: пороговая шкала
* Все баллы блока в 0-100.
* Солнце без пропусков - посчитано для всех 32.
* Цвета и перепад: ожидаем 6 пропусков - Кола и Аджигардак (not_collected, в сценариях скрыты).
* 7 гор обладают 100 баллами по перепаду высоты.
* Максимальное количество баллов в категории "сложность трассы" набрали Роза Хутор и Манжерок.

#### Бинарная шкала
* Расчет для показателей тюбинга и фрирайда.
* Уже 0/1 - переводим в 0/100 умножением, функция не нужна.
* Пропуски ожидаем только у двух неработающих ГК.

In [13]:
scenario_scores['tubing_score'] = mart['tubing'] * 100
scenario_scores['freeride_score'] = mart['freeride'] * 100

print(scenario_scores.shape)
print('тюбинг = 100 у', (scenario_scores['tubing_score'] == 100).sum(), 'курортов')
print('фрирайд = 100 у', (scenario_scores['freeride_score'] == 100).sum(), 'курортов')
print('NaN:', scenario_scores[['tubing_score', 'freeride_score']].isna().sum().sum())

(32, 12)
тюбинг = 100 у 21 курортов
фрирайд = 100 у 23 курортов
NaN: 4


##### Проверка: бинарная шкала
* Только 0 или 100.
* Ожидаемые 4 пропуска из-за статуса работы курорта.

### Финальная проверка: базовые показатели целиком
* Все 13 показателей.
* Диапазон 0-100, пропусков нет - отсутствующая цена получает нейтральные 50 баллов (ТЗ §12)

In [14]:
num = scores.drop(columns='infrastructure_score').select_dtypes('number')
print('Размер:', scores.shape)
print('max:', num.max().max(), '/ min:', num.min().min())
print(num.isna().sum())

summary = pd.DataFrame({
    "min": num.min(),
    "max": num.max(),
    "NaN": num.isna().sum(),
}).round(2)

print(summary)

Размер: (32, 16)
max: 100.0 / min: 0.0
snow_days_score     0
frost_days_score    0
trips_score         0
trails_km_score     0
lifts_score         0
lodging_score       0
food_score          0
snowfall_score      0
transport_score     0
airport_score       0
price_score         0
growth_score        0
stay_score          0
dtype: int64
                    min    max  NaN
snow_days_score    0.00  100.0    0
frost_days_score   0.00  100.0    0
trips_score        0.00  100.0    0
trails_km_score    0.00  100.0    0
lifts_score        0.00  100.0    0
lodging_score      0.00  100.0    0
food_score         0.00  100.0    0
snowfall_score     0.00  100.0    0
transport_score    0.00  100.0    0
airport_score      0.00  100.0    0
price_score        0.00  100.0    0
growth_score       0.00  100.0    0
stay_score        12.47  100.0    0


### Финальная проверка: опциональные показатели целиком
* Все 9 показателей.
* Диапазон 0-100.
* 14 пропусков - по 7 у неработающих ГК.

In [15]:
num = scenario_scores.drop(columns='tr_score').select_dtypes('number')
print('Размер:', scenario_scores.shape)
print('max:', num.max().max(), '/ min:', num.min().min())
print('NaN всего:', num.isna().sum().sum())
print(num.isna().sum())

Размер: (32, 12)
max: 100.0 / min: 0.0
NaN всего: 14
railway_score     0
sun_score         0
green_score       2
blue_score        2
red_score         2
black_score       2
drop_score        2
tubing_score      2
freeride_score    2
dtype: int64


## Итоговый рейтинг

### Функция calculate_rating
* Каждый показатель вносит вклад "балл x вес".
* Вклады складываются и делятся на 100.
* Итог - popularity_score от 0 до 100.
* Равные баллы получают одинаковое место.
* Одна функция считает все рейтинги: базовый и сценарные меняется только словарь весов.

In [16]:
def calculate_rating(score_table, weights):
    # Итог = сумма "балл x вес" по всем показателям, деленная на 100
    # (веса ТЗ - целые числа, в сумме ровно 100).
    rating = score_table.copy()

    total = 0
    for column, weight in weights.items():
        assert rating[column].between(0, 100).all(), 'Проверь баллы: ' + column
        total = total + rating[column] * weight
    rating['popularity_score'] = total / 100

    # Место 1 - лучший итог; равные итоги получают одно место (ТЗ):
    # method='min' - двоим по месту 1, следующий сразу место 3.
    rating['rank'] = rating['popularity_score'].rank(
        ascending=False, method='min'
    ).astype(int)
    return rating

In [17]:
# Мини-тест: три вымышленных курорта и три показателя.
# Цифры круглые - итог легко пересчитать на калькуляторе.
demo_scores = pd.DataFrame({
    'name_ru': ['Снежный', 'Уютный', 'Дешевый'],
    'snow_score': [100, 80, 20],
    'price_score': [40, 60, 100],
    'lift_score': [40, 60, 40],
})

demo_weights = {'snow_score': 50, 'price_score': 30, 'lift_score': 20}

demo_rating = calculate_rating(demo_scores, demo_weights)
demo_rating

,name_ru,snow_score,price_score,lift_score,popularity_score,rank
0,Снежный,100,40,40,70.0,1
1,Уютный,80,60,60,70.0,1
2,Дешевый,20,100,40,48.0,3


##### Вывод по мини-тесту
* Функция считает верно:
  * Итог Снежного проверяем на калькуляторе - 100x50 + 40x30 + 40x20 = 7000, делим на 100 - ровно 70, как в таблице.
  * Снежный и Уютный набрали поровну при разных сильных сторонах - обоим место 1, а следующий сразу место 3: места 2 нет, потому что равные баллы получают одно место (требование ТЗ, в коде - method='min').
  * Дешевый показывает роль весов: идеальная цена не спасает при слабом снеге, у которого вес 50 из 100.

### Базовый рейтинг

#### Таблица весов
* Веса - таблица ТЗ = колонка "вес" в словаре скоринга.
* Веса в сумме дают 100.

In [18]:
tz_weights = {
    'trips_score': 25,       # поездки в регион
    'growth_score': 10,      # рост поездок
    'stay_score': 5,         # длительность поездки
    'trails_km_score': 10,   # километраж трасс
    'lifts_score': 7,        # подъемники
    'lodging_score': 5,      # размещение
    'food_score': 3,         # питание и места интереса
    'snow_days_score': 8,    # снежные дни
    'frost_days_score': 6,   # морозные дни
    'snowfall_score': 6,     # снегопад за сезон
    'airport_score': 8,      # дорога до аэропорта
    'transport_score': 2,    # местный транспорт
    'price_score': 5,        # цена ски-паса
}

assert sum(tz_weights.values()) == 100, 'Веса должны давать в сумме 100'
print('Сумма весов:', sum(tz_weights.values()))

Сумма весов: 100


#### Итоговый рейтинг

In [19]:
base_rating = calculate_rating(scores, tz_weights)

print('Размер:', base_rating.shape)
print('Итог: max', base_rating['popularity_score'].max().round(2),
      '/ min', base_rating['popularity_score'].min().round(2))
print('Пропуски в итоге и местах:',
      base_rating[['popularity_score', 'rank']].isna().sum().sum())

base_rating.sort_values('rank')[['name_ru', 'popularity_score', 'rank']]

Размер: (32, 18)
Итог: max 69.72 / min 28.14
Пропуски в итоге и местах: 0


,name_ru,popularity_score,rank
29,Сорочаны,69.721486,1
2,Газпром Поляна,69.654059,2
0,Роза Хутор,67.241943,3
1,Красная Поляна,65.000503,4
15,Горный воздух,55.783557,5
3,Шерегеш,55.136310,6
4,Архыз,54.700438,7
28,Игора,54.236645,8
5,Домбай,54.181184,9
18,Бобровый лог,51.651806,10


#### Вывод о рейтинге
* Итоговые баллы варьируются от 28.1 до 69.7.
* Это свойство взвешенной суммы: чтобы получить максимум, нужно быть чемпионом сразу по всем 13 показателям, а таких нет.
* Лидер - Сорочаны.
* Места идут с 1 по 32 без склеек - равных итогов не оказалось.

##### Разбор: за что курорт получил место
* Итог складывается из 13 баллов. Чтобы увидеть, какие показатели "тащат" курорт, выводим его баллы по убыванию.
* tz_weights выступает списком колонок рейтинга: берем из строки только их.

In [20]:
# Кто тащит курорт: его 13 баллов по убыванию
for resort_name in ['Сорочаны', 'Аджигардак']:
    row = base_rating[base_rating['name_ru'] == resort_name].iloc[0]
    print(resort_name, '- место', int(row['rank']),
          ', итог', round(row['popularity_score'], 1))
    # astype(float): строка таблицы хранится как object (числа вперемешку
    # с названием курорта), а round умеет только числа - превращаем явно
    print(row[list(tz_weights)].astype(float).sort_values(ascending=False).round(1).to_string())
    print()

Сорочаны - место 1 , итог 69.7
trips_score         100.0
growth_score         99.1
transport_score      77.8
snow_days_score      66.0
lifts_score          63.1
lodging_score        62.0
airport_score        60.3
frost_days_score     55.9
stay_score           55.0
food_score           54.9
price_score          50.0
trails_km_score      33.7
snowfall_score       33.6

Аджигардак - место 13 , итог 48.1
snow_days_score     73.8
frost_days_score    64.9
growth_score        59.5
transport_score     55.2
snowfall_score      51.5
price_score         50.0
trips_score         47.2
trails_km_score     46.3
lifts_score         43.2
food_score          42.8
stay_score          42.6
airport_score       26.6
lodging_score       12.5



##### Вывод по профилям
* Сорочаны берут спросом: поездки и рост почти максимальны при самом большом весе (спрос весит 40 из 100), плюс транспортная доступность Подмосковья. Слабы только километраж трасс и снегопад - это склон у мегаполиса, а не снежная гора.
* Аджигардак место обеспечивает зимняя погода (снег 73.8, мороз 64.9), но размещение 12.5 и аэропорт 26.6 тянут вниз, цена - нейтральные 50. 

#### Таблица достоверности
* Уверенность в данных - отдельное число рядом с рейтингом.
* Оно не входит в итог и не двигает места: у двух курортов с близким итогом качество данных может различаться.
* Формула: confidence = 50 x полнота + 25 x есть цена + 25 x точные координаты

    - полнота = заполненные обязательные поля / 11; состав полей: координаты
      (пара - одно поле), пять полей OSM (еда и места интереса - одной суммой),
      три поля климата, аэропорт, цена
    - точные координаты = точка проверена вручную (official_verified
      или checked_ok)

In [21]:
required_fields = {
    'координаты': ['latitude', 'longitude'],   # пара = одно поле
    'трассы': ['piste_total_km_6km'],
    'лифты': ['lift_lines_6km'],
    'жилье': ['lodging_objects_6km'],
    'еда+интересы': ['food_objects_6km', 'attraction_objects_6km'],
    'транспорт': ['transport_objects_6km'],
    'снежные дни': ['avg_snow_days_per_season'],
    'морозные дни': ['avg_freezing_days_per_season'],
    'снегопад': ['avg_snowfall_cm_per_season'],
    'аэропорт': ['nearest_airport_distance_km'],
    'цена': ['adult_price_min_rub'],
}

# поле заполнено, когда заполнены все его колонки
filled = pd.DataFrame({
    name: mart[cols].notna().all(axis=1) for name, cols in required_fields.items()
})

has_price = mart['adult_price_min_rub'].notna()
exact_coords = mart['coordinate_quality'].isin(['official_verified', 'checked_ok'])

base_rating['confidence_score'] = (
    50 * filled.sum(axis=1) / 11 + 25 * has_price + 25 * exact_coords
).round(2)

print('Распределение confidence_score:')
print(base_rating['confidence_score'].value_counts().to_string())

Распределение confidence_score:
confidence_score
100.00    19
70.45     11
45.45      2


In [22]:
# Кто не набрал 100: поименно, рядом с местом в рейтинге
cols = ['name_ru', 'rank', 'popularity_score', 'confidence_score']
base_rating[base_rating['confidence_score'] < 100][cols].round(2)

,name_ru,rank,popularity_score,confidence_score
0,Роза Хутор,3,67.24,70.45
1,Красная Поляна,4,65.00,70.45
8,Мамисон,31,28.34,70.45
11,Телецкий,28,37.23,70.45
14,КолаСпортЛэнд,21,44.09,45.45
15,Горный воздух,5,55.78,70.45
18,Бобровый лог,10,51.65,70.45
22,Аджигардак,13,48.09,45.45
23,Солнечная долина,24,42.72,70.45
24,Завьялиха,25,42.23,70.45


##### Вывод по достоверности
* У 19 из 32 уверенность полная - 100.
* У 11 уверенность 70.45 - у всех причина одна: нет цены. У 10 из них дневной тариф не публикуется (подъемы, почасовые, пакеты), у Телецкого - реконструкция. Формула бьет дважды: не начисляются 25 пунктов за цену, и полнота падает до 10 полей из 11.
* У КолаСпортЛэнд и Аджигардака 45.45 - нет цены, и координаты не проверяли (обе не работают), теряются еще 25 за координаты.

#### Флаги качества
* Собираем каждому курорту строку предупреждений quality_flags:
  * check coordinates (точка не проверена),
  * coordinates are proxy (точка от ближайшего объекта),
  * OSM radial overlap (радиус пересекается с соседом),
  * source error (ошибка сбора),
  * missing price (цена не найдена, балл 50),
  * цена: прошлый сезон (взята последняя известная цена 25/26),
  * цена: неофициальный источник (не с сайта курорта),
  * не работает (closed),
  * не работает (reconstruction).

In [23]:
# Условия флагов: True = флаг ставится.
# source error не считаем - ошибок сбора не было
flag_conditions = pd.DataFrame({
    'missing price': mart['adult_price_min_rub'].isna(),
    'цена: прошлый сезон': mart['price_season'].eq('2025/26')
                           & mart['adult_price_min_rub'].notna(),
    'цена: неофициальный источник':
            mart['source_confidence'].ne('official_single_source')
            & mart['adult_price_min_rub'].notna(),
    'check coordinates': mart['coordinate_quality'] == 'manual_review_required',
    'coordinates are proxy': mart['coordinate_quality'].str.contains('proxy', na=False),
    'OSM radial overlap': mart['osm_overlap_warning'].notna(),
    'не работает (closed)': mart['operating'] == 'closed',
    'не работает (reconstruction)': mart['operating'] == 'reconstruction',
})

def collect_flags(row):
    # собираем названия флагов, у которых условие True
    active = [name for name in flag_conditions.columns if row[name]]
    return '; '.join(active) if active else 'нет'

# apply(axis=1) идет по таблице строка за строкой
base_rating['quality_flags'] = flag_conditions.apply(collect_flags, axis=1)

In [24]:
for name in flag_conditions.columns:
    print(name, ':', flag_conditions[name].sum())

cols = ['name_ru', 'rank', 'popularity_score', 'confidence_score', 'quality_flags']
base_rating[base_rating['quality_flags'] != 'нет'][cols].sort_values('rank').round(2)

missing price : 13
цена: прошлый сезон : 10
цена: неофициальный источник : 8
check coordinates : 2
coordinates are proxy : 0
OSM radial overlap : 8
не работает (closed) : 1
не работает (reconstruction) : 2


,name_ru,rank,popularity_score,confidence_score,quality_flags
29,Сорочаны,1,69.72,70.45,missing price
2,Газпром Поляна,2,69.65,100.00,цена: прошлый сезон; цена: неофициальный источ...
0,Роза Хутор,3,67.24,70.45,missing price; OSM radial overlap
1,Красная Поляна,4,65.00,70.45,missing price; OSM radial overlap
15,Горный воздух,5,55.78,70.45,missing price
3,Шерегеш,6,55.14,100.00,цена: прошлый сезон
4,Архыз,7,54.70,100.00,цена: прошлый сезон; цена: неофициальный источник
28,Игора,8,54.24,70.45,missing price
5,Домбай,9,54.18,100.00,цена: прошлый сезон; цена: неофициальный источник
18,Бобровый лог,10,51.65,70.45,missing price


##### Вывод по флагам
* Рекордсмен - Аджигардак: три флага сразу (нет цены, точка не проверена, реконструкция) при 13 месте - самый недостоверенный курорт верхней половины.
* Флаг missing price встречается у курортов с отсутствующей ценой на дневной тариф ски-пасса.
* Массовый флаг - "цена: прошлый сезон", у 10 курортов: сезон 26/27 еще не открыли, взята последняя известная цена.
* У Газпрома Поляны теперь три флага (цена за прошлый сезон, неофициальный источник, пересечение радиусов OSM) - при этом уверенность 100: флаги предупреждают, но доверия не отнимают.
* Пересечение радиусов OSM, встречается у 8 курортов, включая лидеров.
* Для Газпрома, Розы и Красной Поляны это не дефект данных, а свойство долины: радиусы 6 км накрывают друг друга, и часть объектов вокруг считается дважды.
* Флаги proxy и source error не сработали ни разу: proxy-точки заменены при ручной проверке координат, ошибок сбора не было.

#### Итоговая таблица базового рейтинга с оценкой уверенности в данных и флагами качества.
* Вся информация о курорте одним взглядом: итог, место, уверенность и флаги.
* Так строка будет выглядеть в итоговой таблице resort_scores и на дашборде.

In [25]:
rating_view = base_rating.sort_values('rank')[
    ['name_ru', 'popularity_score', 'rank', 'confidence_score', 'quality_flags']
].round(2)
rating_view

,name_ru,popularity_score,rank,confidence_score,quality_flags
29,Сорочаны,69.72,1,70.45,missing price
2,Газпром Поляна,69.65,2,100.00,цена: прошлый сезон; цена: неофициальный источ...
0,Роза Хутор,67.24,3,70.45,missing price; OSM radial overlap
1,Красная Поляна,65.00,4,70.45,missing price; OSM radial overlap
15,Горный воздух,55.78,5,70.45,missing price
3,Шерегеш,55.14,6,100.00,цена: прошлый сезон
4,Архыз,54.70,7,100.00,цена: прошлый сезон; цена: неофициальный источник
28,Игора,54.24,8,70.45,missing price
5,Домбай,54.18,9,100.00,цена: прошлый сезон; цена: неофициальный источник
18,Бобровый лог,51.65,10,70.45,missing price


##### Вывод: итоговая таблица
* Вся глава в одной таблице: место, итог, доверие и предупреждения.
* Явное красное пятно - Аджигардак: 13 место при уверенности 45.45 и трех флагах. 

#### Полная проверка согласно ТЗ
* Перед сдачей рейтинга прогоняем чек-лист.
* Каждая проверка печатает результат - провалы видны сразу все, а не по одному.

In [26]:
checks = []

# 1. Веса ТЗ в сумме дают ровно 100
checks.append(('Сумма весов ТЗ = 100', sum(tz_weights.values()) == 100))

# 2. В рейтинге 32 строки и 32 уникальных курорта
checks.append(('32 строки, 32 уникальных id',
               len(base_rating) == 32 and base_rating['resort_id'].nunique() == 32))

# 3. Итог и место без пропусков
checks.append(('Итог и место без пропусков',
               base_rating[['popularity_score', 'rank']].isna().sum().sum() == 0))

# 4. Нет бесконечностей в итоге
checks.append(('Нет бесконечностей в итоге',
               not np.isinf(base_rating['popularity_score']).any()))

# 5. Итог в разумном диапазоне 0-100
checks.append(('Итог в диапазоне 0-100',
               base_rating['popularity_score'].between(0, 100).all()))

for name, ok in checks:
    print(('OK      ' if ok else 'ПРОВАЛ  ') + name)

OK      Сумма весов ТЗ = 100
OK      32 строки, 32 уникальных id
OK      Итог и место без пропусков
OK      Нет бесконечностей в итоге
OK      Итог в диапазоне 0-100


##### Вывод по полной проверке
* Пять из пяти - рейтинг принят по ТЗ: веса честные, все 32 курорта на месте, итоги без пропусков и аварий. 

#### Стресс-тест весов
* Проверяем устойчивость: что будет, если одной группе показателей придать на 20% больше важности?
* Каждой из пяти групп по очереди увеличиваем веса на 20%, остальные пропорционально сжимаем - сумма весов остается ровно 100.
* Рейтинг пересчитывается, сравниваем места с базовыми.
* Вес группы = сумма весов ТЗ входящих в нее показателей: спрос 25+10+5=40, зима 8+6+6=20, инфраструктура 10+7+5+3=25, доступность 8+2=10, цена 5. Вместе - те же 100.

In [27]:
weight_groups = {
    'спрос': ['trips_score', 'growth_score', 'stay_score'],
    'зима': ['snow_days_score', 'frost_days_score', 'snowfall_score'],
    'инфраструктура': ['trails_km_score', 'lifts_score', 'lodging_score', 'food_score'],
    'доступность': ['airport_score', 'transport_score'],
    'цена': ['price_score'],
}

# группы обязаны накрывать все 13 показателей ровно один раз
all_grouped = [col for cols in weight_groups.values() for col in cols]
assert sorted(all_grouped) == sorted(tz_weights), 'Группы должны покрывать все 13 показателей'

for group_name, cols in weight_groups.items():
    print(group_name, '-', sum(tz_weights[c] for c in cols))

спрос - 40
зима - 20
инфраструктура - 25
доступность - 10
цена - 5


In [28]:
def stress_weights(group_cols):
    # группа +20%, остальные пропорционально сжимаем, сумма остается 100
    group_sum = sum(tz_weights[c] for c in group_cols)
    other_scale = (100 - 1.2 * group_sum) / (100 - group_sum)
    weights = {}
    for col, weight in tz_weights.items():
        if col in group_cols:
            weights[col] = weight * 1.2
        else:
            weights[col] = weight * other_scale
    assert round(sum(weights.values()), 6) == 100, 'Сумма весов должна остаться 100'
    return weights

stress_ranks = pd.DataFrame({'name_ru': base_rating['name_ru'],
                             'база': base_rating['rank']})
for group_name, group_cols in weight_groups.items():
    variant = calculate_rating(scores, stress_weights(group_cols))
    stress_ranks['+' + group_name] = variant['rank'].values

stress_ranks

,name_ru,база,+спрос,+зима,+инфраструктура,+доступность,+цена
0,Роза Хутор,3,3,3,2,3,3
1,Красная Поляна,4,4,4,4,4,4
2,Газпром Поляна,2,2,2,1,2,2
3,Шерегеш,6,9,6,5,6,6
4,Архыз,7,6,8,7,7,7
5,Домбай,9,7,7,8,9,9
6,Эльбрус,12,18,12,12,13,14
7,Чегет,16,20,14,13,16,15
8,Мамисон,31,31,32,31,31,32
9,Цей,32,32,31,32,32,31


In [29]:
# изменение мест в каждом сценарии (по модулю)
diff = (stress_ranks.drop(columns=['name_ru', 'база']) - stress_ranks[['база']].values).abs()

# худший прыжок курорта по всем пяти сценариям
jump = diff.max(axis=1)

# приклеиваем имена к прыжкам: собираем мини-таблицу из двух колонок
jump_named = pd.DataFrame({'name_ru': stress_ranks['name_ru'], 'прыжок': jump})
jump_named.sort_values('прыжок', ascending=False).head(10)

,name_ru,прыжок
6,Эльбрус,6
12,Большой Вудъявр,5
20,Гора Белая,5
14,КолаСпортЛэнд,4
17,Морозная гора,4
7,Чегет,4
3,Шерегеш,3
15,Горный воздух,3
25,Абзаково,3
31,Гладенькая,3


##### Вывод: стресс-тест
* Состав топ-4 одинаков во всех пяти сценариях: Сорочаны, Газпром Поляна, Роза Хутор, Красная Поляна. Меняется только порядок: при +20% к инфраструктуре Газпром выходит на 1-е место, Роза - на 2-е, Сорочаны - на 3-е (в базе отрыв лидера всего 0.07 балла).
* Красная Поляна - единственный курорт с абсолютно неподвижным местом: 4-е во всех шести колонках.
* В топ-10 максимальный сдвиг - 3 места, и весь в сценарии +спрос: Горный воздух 5 -> 8 и Шерегеш 6 -> 9 падают, навстречу поднимаются Игора 8 -> 5 и Домбай 9 -> 7.
* Сильнее всего перетасовывает региональный спрос: пять из шести максимальных прыжков таблицы случаются в колонке +спрос. Горы без спроса падают (Эльбрус 12 -> 18, Чегет 16 -> 20, Большой Вудъявр 11 -> 16), курорты в крупных регионах растут (Гора Белая 20 -> 15). Единственное исключение - Морозная гора 15 -> 11 при +доступности: близкий аэропорт крупного города.
* Колонка +цена после ревизии не меняет ни одного места в топ-10, максимум по таблице - 2 места. Цена весит 5%, известна для 19 курортов из 32, у остальных нейтральный балл - усиление ее веса ничего не решает. Прямой ответ на вопрос "не перевернет ли цену пересмотр тарифов" - нет.
* Низ рейтинга стабилен: Гладенькая 30-е, Холдоми 29-е при любых весах.
Вывод: веса можно двигать в пределах 20% - состав головы не меняется; единственная по-настоящему чувствительная группа - региональный спрос.

### Вывод по базовому рейтингу
* Рейтинг посчитан: 32 курорта, места 1-32 без склеек, полная проверка по ТЗ.
* Итоговые баллы находятся в пределах от 28.1 до 69.7 - это свойство взвешенной суммы: лидером сразу по всем 13 показателям быть нельзя.
* Топ-5: Сорочаны, Газпром Поляна, Роза Хутор, Красная Поляна, Горный воздух.
* Топ курорта Сорочаны обеспечивается спросом - рядом крупный город и доступность;
* У краснодарской тройки двойной источник силы - спрос региона и развитая инфраструктура;
* Горный воздух - зимняя погода и близость к городу;
* popularity_score - это популярность, а не "лучшая гора". №1 страны - склон у мегаполиса, а не снежная вершина.
* Сценарии следующей главы покажут эту же таблицу глазами нескольких персонажей - картина будет другой.
* Достоверность: у 19 из 32 полная, у 11 нет цены (см. таблицу достоверности).
* Устойчивость: стресс-тест +20% к каждой группе весов не меняет состав топ-4 ни в одном сценарии (порядок внутри меняется при усилении инфраструктуры), в топ-10 сдвиг максимум 3 места.

### Сценарный рейтинг
* Сценарный рейтинг - те же баллы, но с разными весами под разные сценарии.
* Трое персонажей:
  * хардкор-лыжник
  * семья с детьми
  * отдыхающий без лыж
* Показываем только 29 работающих курортов.

#### Сценарные веса

| Показатель | Хардкор-лыжник | Семья | Отдыхающий без лыж |
|---|---|---|---|
| Перепад высот курорта (общий перепад зоны катания) | 25 | - | - |
| Снежные дни | 20 | - | - |
| Черные трассы | 15 | - | - |
| Фрирайд | 10 | - | - |
| Снегопад | 10 | - | - |
| Подъемники | 10 | - | - |
| Красные трассы | 10 | - | - |
| Зеленые трассы | - | 25 | - |
| Тюбинг | - | 20 | - |
| ЖД-станция | - | 20 | - |
| Цена | - | 15 | 15 |
| Питание и места интереса | - | 10 | 25 |
| Размещение | - | 5 | 20 |
| Аэропорт | - | 5 | 15 |
| Солнечные часы | - | - | 25 |

Сумма весов каждого персонажа - 100. Базовые показатели из словаря выше
участвуют у персонажей по мере нужд человека.

In [30]:
scenario_matrix = scores.merge(scenario_scores, on=['resort_id', 'name_ru'],
                               validate='one_to_one')
scenario_matrix = scenario_matrix.merge(mart[['resort_id', 'operating']],
                                        on='resort_id', validate='one_to_one')
scenario_matrix = scenario_matrix[scenario_matrix['operating'] == 'yes'] \
    .drop(columns='operating').reset_index(drop=True)

print('Работающих курортов в сценариях:', len(scenario_matrix))
print('Пропусков:', int(scenario_matrix.isna().sum().sum()))

Работающих курортов в сценариях: 29
Пропусков: 0


#### Персонаж 1. Хардкор-лыжник
Ездит за снегом и рельефом: перепад высоты, черные трассы, внетрассовое. Цена и дорога почти не важны.

In [31]:
hardcore_weights = {
    'drop_score': 25,        # перепад - главный экстрим
    'snow_days_score': 20,   # снег должен быть
    'black_score': 15,       # черные трассы
    'freeride_score': 10,    # внетрассовое катание
    'snowfall_score': 10,    # снегопад за сезон
    'lifts_score': 10,       # подъемники
    'red_score': 10,         # красные трассы
}
assert sum(hardcore_weights.values()) == 100

hardcore_rating = calculate_rating(scenario_matrix, hardcore_weights)
hardcore_rating.sort_values('rank')[['name_ru', 'popularity_score', 'rank']].head(5)

,name_ru,popularity_score,rank
7,Чегет,88.670376,1
2,Газпром Поляна,80.507940,2
11,Большой Вудъявр,76.804390,3
5,Домбай,76.611709,4
10,Манжерок,76.505873,5


In [32]:
# чем набрал лидер: вклад каждого показателя = балл x вес / 100
top_name = hardcore_rating.sort_values('rank')['name_ru'].iloc[0]
w = pd.Series(hardcore_weights)
row = scenario_matrix.set_index('name_ru').loc[top_name, w.index].astype(float)
contrib = (row * w / 100).sort_values(ascending=False)
print(top_name)
print(contrib.round(1))

Чегет
drop_score         25.0
snow_days_score    17.7
black_score        15.0
freeride_score     10.0
snowfall_score      8.9
red_score           7.0
lifts_score         5.1
dtype: float64


##### Вывод: хардкор-лыжник
Лидер - Чегет, 88.7: перепады высоты, снежные дни, черные трассы. В базовом рейтинге он 16-й. Сорочаны из топа исчезают совсем:
мегаполис не помогает, когда нужны перепад и черные трассы. 

#### Персонаж 2. Семья с детьми
Новичковые зеленые трассы, тюбинг, поезд вместо самолета, бюджет
и где поесть с детьми.

In [33]:
family_weights = {
    'green_score': 25,       # зеленые трассы для новичков
    'tubing_score': 20,      # тюбинг
    'railway_score': 20,     # поезд без перелета с детьми
    'price_score': 15,       # бюджет
    'food_score': 10,        # поесть с детьми
    'lodging_score': 5,
    'airport_score': 5,
}
assert sum(family_weights.values()) == 100

family_rating = calculate_rating(scenario_matrix, family_weights)
family_rating.sort_values('rank')[['name_ru', 'popularity_score', 'rank']].head(5)

,name_ru,popularity_score,rank
0,Роза Хутор,79.506591,1
1,Красная Поляна,76.412583,2
2,Газпром Поляна,72.861771,3
13,Горный воздух,65.104392,4
16,Бобровый лог,61.389715,5


##### Вывод: семья
Краснодарская тройка впереди: зеленые трассы, тюбинг, олимпийская ЖД-ветка,
еда вокруг. Горный воздух и Бобровый лог рядом - тюбинг и поезд.

#### Персонаж 3. Отдыхающий без лыж
* Едет в горы не кататься: акцент на солнце, проживании и питании.
* Трассы не важны совсем - в весах нет ни одного трассового показателя.
* Цена - прокси показатель общей ценовой доступности региона, а не билет на подъемник: человек едет не в абы какие горы, а туда, где красиво и развито - и дорогой курорт означает дорогой регион в целом (еда, жилье вокруг).

In [34]:
comfort_weights = {
    'sun_score': 25,        # солнышко и тепло - главное
    'food_score': 25,       # хорошая еда вокруг
    'lodging_score': 20,    # комфортное жилье
    'price_score': 15,
    'airport_score': 15,    # добраться без подвига
}
assert sum(comfort_weights.values()) == 100

comfort_rating = calculate_rating(scenario_matrix, comfort_weights)
comfort_rating.sort_values('rank')[['name_ru', 'popularity_score', 'rank']].head(5)

,name_ru,popularity_score,rank
1,Красная Поляна,86.499698,1
13,Горный воздух,84.379894,2
0,Роза Хутор,78.057952,3
2,Газпром Поляна,77.821321,4
16,Бобровый лог,74.035969,5


##### Вывод: отдыхающий без лыж
Лидер Красная Поляна 86.5 - кавказское солнце и развитая инфраструктура. Горный воздух 84.4 второй - солнце, транспорт и инфраструктура. 

In [35]:
summary = pd.DataFrame({
    'база': base_rating.sort_values('rank')['name_ru'].head(5).tolist(),
    'хардкор': hardcore_rating.sort_values('rank')['name_ru'].head(5).tolist(),
    'семья': family_rating.sort_values('rank')['name_ru'].head(5).tolist(),
    'комфорт': comfort_rating.sort_values('rank')['name_ru'].head(5).tolist(),
})
summary

,база,хардкор,семья,комфорт
0,Сорочаны,Чегет,Роза Хутор,Красная Поляна
1,Газпром Поляна,Газпром Поляна,Красная Поляна,Горный воздух
2,Роза Хутор,Большой Вудъявр,Газпром Поляна,Роза Хутор
3,Красная Поляна,Домбай,Горный воздух,Газпром Поляна
4,Горный воздух,Манжерок,Бобровый лог,Бобровый лог


### Вывод по сценарным рейтингам
Один и тот же набор баллов дает четыре разные картины: база - Сорочаны,
хардкор-лыжник - Чегет, семья - Роза Хутор, отдыхающий без лыж -
Красная Поляна. Лидер меняется вместе
с разными сценариями - это и есть ответ на вопрос "зачем сценарии": базовый рейтинг
показывает среднюю температуру, вымышленные персонажи - конкретные потребности.
Сценарий ТЗ "большой вес климата" закрыт персонажем хардкор-лыжника,
"большой вес доступности" посчитан в стресс-тесте (колонка "+доступность"
таблицы мест).

## Ответы на вопросы для исследования согласно ТЗ

### Вопрос 1. Какие три показателя сильнее всего влияют на итоговый порядок?
Влияние меряем ранговой корреляцией: сравниваем порядок курортов по показателю с порядком по итогу. Корреляция 1 - показатель полностью определяет итог, 0 - порядок показателя не связан с итоговым, минус - тянет в обратную сторону.

In [36]:
influence = pd.Series({
    col: scores[col].rank().corr(base_rating['popularity_score'].rank())
    for col in tz_weights
})
influence.sort_values(ascending=False).round(2)

lifts_score         0.76
food_score          0.73
lodging_score       0.71
transport_score     0.60
trails_km_score     0.47
trips_score         0.47
growth_score        0.40
stay_score          0.35
airport_score       0.35
snowfall_score      0.21
snow_days_score    -0.06
price_score        -0.17
frost_days_score   -0.32
dtype: float64

**Ответ:** 
* Три показателя, которые сильнее всего влияют на итоговый порядок: подъемники (0.76), питание и места интереса (0.73), размещение (0.71). Все три - инфраструктура.
* Поездки имеют самый большой вес (25), но влияют умеренно (0.47), потому что спрос считается по региону и не различает курорты внутри региона.
* Самые слабые связи - у морозных дней (-0.32) и цены (-0.17), потому что у 13 курортов цена - нейтральная заглушка 50, а морозные курорты в среднем уступают по инфраструктуре и спросу.

### Вопрос 2. Какие курорты находятся высоко из-за регионального спроса, а какие - благодаря климату или инфраструктуре?
* Вклад группы в итог курорта = сумма "балл x вес" ее показателей / 100.
* Вклады пяти групп складываются ровно в popularity_score - можно разложить итог любого курорта по группам.

In [37]:
group_contrib = pd.DataFrame({'name_ru': base_rating['name_ru']})
for group_name, group_cols in weight_groups.items():
    group_contrib[group_name] = sum(
        scores[col] * tz_weights[col] for col in group_cols) / 100

groups = list(weight_groups)          # пять имен групп
group_contrib['итог'] = group_contrib[groups].sum(axis=1)
group_contrib['место'] = base_rating['rank'].values
view = group_contrib.set_index('name_ru').sort_values('место').round(2)
view

,спрос,зима,инфраструктура,доступность,цена,итог,место
name_ru,,,,,,,
Сорочаны,37.66,10.65,12.53,6.38,2.50,69.72,1
Газпром Поляна,32.36,8.79,21.95,5.78,0.77,69.65,2
Роза Хутор,32.36,2.95,23.52,5.91,2.50,67.24,3
Красная Поляна,32.36,0.44,23.46,6.24,2.50,65.00,4
Горный воздух,14.14,13.45,15.85,9.84,2.50,55.78,5
Шерегеш,15.46,15.58,20.93,3.03,0.13,55.14,6
Архыз,20.59,9.69,19.18,3.14,2.10,54.70,7
Игора,27.32,10.23,9.93,4.25,2.50,54.24,8
Домбай,20.59,16.68,14.98,1.92,0.00,54.18,9


**Ответ:** Сорочаны: 37.7 из 70 - спрос принес больше половины итоговых баллов.
Краснодарская тройка: спрос одинаковый 32.4 (регион общий) - их взаимный порядок решают зима и инфраструктура. 
Шерегеш и Горный воздух - "инфраструктурные" горы: спрос средний, но 20.9 и 15.9 от инфраструктуры тащат в топ-6.
Эльбрус и Чегет - "климатические": зима 18.5 и 17.6 - лучшие цифры всей таблицы, но спрос 9.5 - поэтому середина.

### Вопрос 3. У каких объектов высокий индекс сочетается с низкой уверенностью?

**Ответ:** Аджигардак - 13 место при уверенности 45.45 и трех флагах
(нет дневной цены, точка не проверена, реконструкция). Единственный курорт
верхней половины с неполными данными. КолаСпортЛэнд тоже с 45.45,
но она на 21 месте.

### Вопрос 4. Какие места меняются при увеличении каждого группового веса на 20 процентов?

**Ответ:** Состав топ-4 не меняется ни в одном из пяти сценариев (+20% к группе), меняется только порядок при усилении инфраструктуры. Сильнее всего перетасовывает середину региональный спрос (до 6 мест); усиление цены не меняет ни одного места в топ-10.
Считалось в разделе "Стресс-тест весов".


### Вопрос 5. Есть ли пары соседних курортов, для которых OSM считает одни и те же объекты?

**Ответ:** да, три группы - восемь курортов с флагом OSM radial overlap:
краснодарская тройка (Роза Хутор, Газпром, Красная Поляна - расстояния
4.8-6.6 км при радиусе сбора 6 км), пара Эльбрус-Чегет (5.7 км)
и кировская группа Большой Вудъявр - Кукисвумчорр - КолаСпортЛэнд
(3.1-7.5 км). Часть объектов вокруг у них считается дважды -
флаг честно предупреждает об этом, не отнимая доверия.
Считалось в разделе "Флаги качества".

### Вопрос 6. Как расширенные показатели Росстата меняют первые десять мест?

**Ответ:** слой не подключали - ТЗ (раздел "Дополнительный слой Росстата")
прямо называет его необязательным для первой версии рейтинга. Для него нужны
данные КСР по регионам: размещенные туристы, выручка и зимняя доля
размещений. Это задел на развитие: при появлении данных показатели
добавляются в витрину и в веса без перестройки рейтинга.

### Вопрос 7. Каких данных не хватает, чтобы говорить о фактическом спросе на отдельный курорт?

**Ответ:** все наши данные спроса - региональные: Росстат считает субъекты,
курортных рядов не существует, и три курорта одного региона получают
одинаковые значения. Для фактического спроса
на конкретный курорт не хватает: продаж ски-пассов, загрузки отелей
у склона, билетов до ближайшей станции или аэропорта, поисковых запросов
и бронирований по названию курорта. .